In [129]:
#import cell
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn, optim
from torch.nn import functional as F
from torch.utils.tensorboard import SummaryWriter
import matplotlib.pyplot as plt
import copy
from tqdm import tqdm
import csv
from scipy.special import softmax
device = torch.device('cuda:0')


In [132]:
#cell for network architecture

class DataSet():
    def __init__(self,
                 ground_truth_path=None,
                 sample_size=5,
                 batch_size=1,
                 max_data_size=10000):
        self.feature_names = []
        self.max_data_size = max_data_size
        self.sample_size=sample_size
        self.batch_size=batch_size
        if ground_truth_path is not None:
            self.groundtruth_data = self.load_csv(ground_truth_path)
            self.n_truth = self.groundtruth_data.shape[0]
            self.groundtruth_weights = np.arange(self.n_truth)/self.n_truth
            self.groundtruth_weights = softmax(self.groundtruth_weights)
            self.biased_data = self.groundtruth_data
            self.n_biased = self.biased_data.shape[0]
            self.biased_weights = np.zeros(self.n_biased)
            self.x_dim = self.groundtruth_data.shape[1]
        else: #dummy dataset
            self.groundtruth_data = np.eye(10)
            self.n_truth = self.groundtruth_data.shape[0]
            self.groundtruth_weights =np.zeros(self.n_truth)
            self.biased_data = self.groundtruth_data
            self.n_biased = self.biased_data.shape[0]
            self.x_dim = self.groundtruth_data.shape[1]
            self.biased_weights = np.zeros(self.n_biased)
            
            for i in range(self.groundtruth_data.shape[0]):
                self.groundtruth_weights[i] = 0
                self.biased_weights[i] = 1/self.x_dim
                self.groundtruth_weights[0] = 0.4
                self.groundtruth_weights[1] = 0.3
                self.groundtruth_weights[2] = 0.2
                self.groundtruth_weights[3] = 0.1

    def load_csv(self,data_path):
        with open(data_path, newline='\n') as csvfile:
            data_count = 0
            spamreader = csv.reader(csvfile, delimiter=',', quotechar='"')
            data = []
            for row_count, row in tqdm(enumerate(spamreader)):
                if row_count == 0:
                    self.feature_names = row
                else:
                    cur_data = np.zeros(len(row),dtype=float)
                    for did, datum in enumerate(row):
                        if datum == '':
                            cur_data[did] = -1.0
                        else:
                            cur_data[did] = float(datum)
                    data.append(cur_data)
                    data_count += 1
                    if data_count >= self.max_data_size:
                        break
        data = np.array(data)
        return data

    def get_groundtruth_batch(self):
        batch = []
        for _ in range(self.batch_size): 
            groundtruth_indexes = np.random.choice(a=self.n_truth,size=self.sample_size,p=self.groundtruth_weights)
            batch.append(self.groundtruth_data[groundtruth_indexes].flatten())
        batch = np.stack(batch)
        return batch
    
class generator_noNN(torch.nn.Module):
    def __init__(self,
                 num_classes,
                 subset_size,
                 tau,
                 batch_size,
                 device):
        super().__init__()
        self.subset_size=subset_size
        self.num_classes=num_classes
        #self.weights = torch.ones(num_classes,requires_grad=True).to(device)
        self.weights = torch.rand((1,num_classes), requires_grad=True, device=device)
        self.tau = tau
        self.batch_size=batch_size

    def forward(self,tensor_data):
        batch = []
        for _ in range(self.batch_size):
            indexes = F.gumbel_softmax(self.weights.repeat(self.subset_size,1), 
                                       tau=self.tau, 
                                       hard=False) #picks from the logits
            batch.append((indexes @ tensor_data).flatten())
        return torch.stack(batch,axis=0)


class Generator(torch.nn.Module):
    def __init__(self,
                 batch_size,
                 temperature,
                 device):
        super(Generator, self).__init__()
        self.linear = nn.Linear(1, 5)
        self.batch_size=batch_size
        self.temperature = temperature
        self.device=device

    def forward_notworking(self):
        input = torch.tensor([[1] for _ in range( self.batch_size)], dtype=torch.float32).to(device)
        input = input.reshape( self.batch_size, 1)
        logits = self.linear(input)
        output = F.gumbel_softmax(logits, tau=self.temperature, hard=False) #picks from the logits
        return output
    

    def test(self):
        input = torch.tensor([[1]], dtype=torch.float32).to(self.device)
        logits = self.linear(input)
        output = nn.Softmax(dim=1)(logits)
        return output

#latent_dim = 1
categorical_dim = 10  # one-of-K vector

def sample_gumbel(shape, eps=1e-20):
    U = torch.rand(shape)
    U = U.to(device)
    return -torch.log(-torch.log(U + eps) + eps)
def gumbel_softmax_sample(logits, temperature):
    y = logits + sample_gumbel(logits.size())
    return F.softmax(y, dim=-1)

def gumbel_softmax(logits, tau, hard=False):
    """
    ST-gumple-softmax
    input: [*, n_class]
    return: flatten --> [*, n_class] an one-hot vector
    """
    print("self gumbel softmax")
    y = gumbel_softmax_sample(logits, tau)
    if not hard:
        return y.view(-1, categorical_dim)

    shape = y.size()
    _, ind = y.max(dim=-1)
    y_hard = torch.zeros_like(y).view(-1, shape[-1])
    y_hard.scatter_(1, ind.view(-1, 1), 1)
    y_hard = y_hard.view(*shape)
    # Set gradients w.r.t. y_hard gradients w.r.t. y
    y_hard = (y_hard - y).detach() + y
    return y_hard.view(-1, categorical_dim)

class DataGenerator(torch.nn.Module):
    def __init__(self,
                 subset_size,
                 batch_size,
                 temperature,
                 output_size,
                 device):
        super(DataGenerator, self).__init__()
        self.linear = nn.Linear(1, output_size).to(device)
        self.batch_size=batch_size
        self.subset_size=subset_size
        self.temperature = temperature
        self.device=device

    def forward(self,tensor_dataset):
        '''
        dataset - is the data set cast as a torch tensor and moved to the appropriate device.

        samples indexes using gumbel softmax + params

        matrix multiples the one-hot-encoded indexes with data set
        to differentiably isolate SUBSET_SIZE data points
        '''
        
        batch = []
        for _ in range(self.batch_size):
            input = torch.tensor([[1] for _ in range(self.subset_size)], dtype=torch.float32).to(device)
            input = input.reshape(self.subset_size, 1)
            logits = self.linear(input)
            indexes = F.gumbel_softmax(logits, tau=self.temperature, hard=False) #picks from the logits
            output = indexes @ tensor_dataset
            batch.append(output.flatten())
        return torch.stack(batch,axis=0), indexes.detach().cpu().numpy()
    
    def forward_debug(self,tensor_dataset):
        '''
        dataset - is the data set cast as a torch tensor and moved to the appropriate device.

        samples indexes using gumbel softmax + params

        matrix multiples the one-hot-encoded indexes with data set
        to differentiably isolate SUBSET_SIZE data points
        '''
        
        for _ in range(self.batch_size):
            input = torch.tensor([[1] for _ in range(self.subset_size)], dtype=torch.float32).to(device)
            input = input.reshape(self.subset_size, 1)
            logits = self.linear(input)
            indexes = F.gumbel_softmax(logits, tau=self.temperature, hard=False) #picks from the logits
            output = indexes @ tensor_dataset
            
        return output.flatten().unsqueeze(0)
    
    def get_weights(self):
        input = torch.tensor([[1]], dtype=torch.float32).to(device)
        logits = self.linear(input)
        softmaxed = torch.nn.functional.softmax(logits,dim=1)
        return softmaxed


    def test(self):
        input = torch.tensor([[1]], dtype=torch.float32).to(self.device)
        with torch.no_grad():
            logits = self.linear(input)
            output = nn.Softmax(dim=1)(logits)
        return output

class DataDisctiminator(torch.nn.Module):
    def __init__(self,
                 subset_size,
                 data_dimension,):
        '''
        1. maybe discriminator is too weak (in terms of expressability)
        2. generator should be slowed down in terms of the discriminator
        3. look at tabular GAN papers for how they structure discriminator
        '''
        super(DataDisctiminator, self).__init__()
        self.linear = nn.Linear(subset_size*data_dimension, 1)
        self.linear.to(device)

    def forward(self, net_input):
        output = self.linear(net_input)
        return output

class Disctiminator(torch.nn.Module):
    def __init__(self):
        super(Disctiminator, self).__init__()
        self.linear = nn.Linear(5, 1)

    def forward(self, input):
        output = self.linear(input)
        return output

def mlp(
    input_size,
    layer_sizes,
    output_size,
    output_activation=torch.nn.Identity,
    activation=torch.nn.ELU,
):
    sizes = [input_size] + layer_sizes + [output_size]
    layers = []
    for i in range(len(sizes) - 1):
        act = activation if i < len(sizes) - 2 else output_activation
        layers += [torch.nn.Linear(sizes[i], sizes[i + 1]), act()]
    return torch.nn.Sequential(*layers)


In [145]:

def to_onehot(y, num_classes=5):
    return np.eye(num_classes)[y].reshape(-1, num_classes)

def train(dataset,generator,generator_optim,discriminator,discriminator_optim,L):
    learned_probs = []
    generator.train()
    discriminator.train()
    tensor_data = torch.tensor(dataset.biased_data,dtype=torch.float).to(device)
    for epoch in tqdm(range(EPOCHS)):
        #get ground truth data from data set
        ground_truth_data = torch.tensor(dataset.get_groundtruth_batch(),dtype=torch.float).to(device)
        ground_truth_prediction= discriminator(ground_truth_data)

        #get biased data
        bias_data, _ = generator(tensor_data)
        bias_prediction = discriminator(bias_data)

        #compute truth loss
        loss_real = L(ground_truth_prediction.to(device), torch.ones_like(ground_truth_prediction).to(device))
        #compute fake loss
        loss_fake = L(bias_prediction.to(device), torch.zeros_like(bias_prediction).to(device))
        #combine losses
        discriminator_loss = loss_real+loss_fake
        if epoch % WRITE_EVERY == 0:
            writer.add_scalar("Discriminator Loss: ", discriminator_loss.detach().cpu().item(), epoch)
        #update discriminator
        discriminator_optim.zero_grad()
        discriminator_loss.backward(retain_graph=True)
        discriminator_optim.step()
        for _ in range(GENERATOR_TRAINING_FACTOR):
            # update generator
            bias_data_gen, _ = generator(tensor_data)
            bias_prediction_gen = discriminator(bias_data_gen)
            generator_loss = L(bias_prediction_gen, torch.ones_like(bias_prediction_gen).to(device)) + SPREAD_LAMBDA*generator.get_weights().std()
            generator_optim.zero_grad()
            generator_loss.backward()
            generator_optim.step()
            #print(list(generator.linear.parameters()))
            if epoch % WRITE_EVERY == 0:
                writer.add_scalar("Generator Loss: ", generator_loss.detach().cpu().item(), epoch)
        writer.flush()
    return learned_probs, saved_graph

#Variable Setup
saved_graph = None
DATA_DIM = 10 #dimentionality of each data point
SUBSET_SIZE = 30 #the "sample size" of each data point. 
BATCH_SIZE = 1 #the number of subsets to be passed to the discriminator
EPOCHS = 1000  # the stream is infinite so one epoch will be defined as BATCHS_IN_EPOCH * BATCH_SIZE
GENERATOR_TRAINING_FACTOR = 10  # for every training of the disctiminator we'll train the generator 10 times
LEARNING_RATE = 1e-2
WRITE_EVERY = EPOCHS/100 #frequency fo save loss values to file
writer = SummaryWriter(comment="dummyDataSet")

TEMPERATURE = 0.3
WEIGHT_DECAY = 1e-3
SPREAD_LAMBDA = 0 #10

stored_updates = []
dataset = DataSet(ground_truth_path= None, #"./data/usa_00002.csv",
                  sample_size=SUBSET_SIZE,
                batch_size=BATCH_SIZE)
generator = DataGenerator(subset_size=SUBSET_SIZE,
                            batch_size=BATCH_SIZE,
                            temperature=TEMPERATURE,
                            output_size=dataset.n_biased,
                            device=device)
generator.train()
generator_optimizer = torch.optim.Adam(generator.parameters(), lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
discriminator = DataDisctiminator(subset_size=SUBSET_SIZE,data_dimension=DATA_DIM)
discriminator_optimizer = torch.optim.Adam(discriminator.parameters(), lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
L = nn.BCEWithLogitsLoss()
if False:
    store_params = []
    for p in generator.linear.parameters():
        store_params.append(copy.deepcopy(p.detach().cpu()))
        #print(p)
    ground_truth_data = torch.tensor(dataset.get_groundtruth_batch(),dtype=torch.float).to(device)
    tensor_data = torch.tensor(dataset.biased_data,dtype=torch.float).to(device)
    bias_data_gen = generator(tensor_data)
    #print("generated data: ", bias_data_gen.shape)
    bias_prediction_gen = discriminator(bias_data_gen)
    generator_loss = L(bias_prediction_gen, torch.ones_like(bias_prediction_gen).to(device))
    print("gen loss: ", generator_loss.detach().cpu().item())
    generator_optimizer.zero_grad()
    generator_loss.backward()
    generator_optimizer.step()
    for i,p in enumerate(list(generator.linear.parameters())):
        print(i,(p.detach().cpu() != store_params[i]).all().item())
        #print(p)

start_weights = generator.get_weights().detach().cpu().numpy().flatten()
print(start_weights[-10:])
print("Starting diff (sum): ",  np.linalg.norm(start_weights - dataset.groundtruth_weights,ord=2))
if True:
    _, saved_graph = train(dataset,generator,generator_optimizer,discriminator,discriminator_optimizer,L)
    tensor_data = torch.tensor(dataset.biased_data,dtype=torch.float).to(device)
    generator.eval()
    generator.batch_size = 1
    with torch.no_grad():
        eval_output, indexes = generator(tensor_data)
        eval_output = torch.reshape(eval_output,(SUBSET_SIZE,DATA_DIM)).cpu().numpy()
    #print(eval_output)
    print("Done")

learned_weights = generator.get_weights().detach().cpu().numpy().flatten()
print("ending diff (sum): ", np.linalg.norm(learned_weights - dataset.groundtruth_weights,ord=2))
print(learned_weights[-10:])

[0.23600566 0.05246322 0.04847214 0.03120611 0.13025655 0.10666335
 0.08270501 0.03648425 0.13781214 0.13793166]
Starting diff (sum):  0.4363419297215537


100%|██████████| 1000/1000 [00:47<00:00, 20.98it/s]

Done
ending diff (sum):  0.6831868265624247
[9.8030829e-01 1.3553040e-02 4.5371656e-03 1.3646428e-03 3.9250994e-05
 3.6021553e-05 3.6637728e-05 4.1536579e-05 3.7932365e-05 4.5401215e-05]


Keep the temperature > 0.1, < 0.1 it has adverse effects and sometimes doesnt update the weights. probablyt due to the fact that the logits are divided by the temperature. and thus it creates some type of instability

Weight decay:
    increasing the weight decay causes the network weights to become more uniform, but does not solve the issue where 

Another idea: std of the probability distribution. kinda crude, and maybe not applicable to larger scale data.
lambda = 1e-3 no effect
lambda = 1e-1, 0.1 + 0.9
lambda = 2e-1, 0.05 v 0.95
lambda = 3e-1, 0.1 v 0.9, 0.05 v 0.95
lambda = 1, 0.06 vs 0.94
lambda = 2.5, 0.46 vs 0.53
lambda = 5, 0.5 vs 0.49
lambda = 10, 0.5 vs 0.5

truth [ 0.4, 0.3, 0.2, 0.1, 0...]
lambda = 2.5 - [0.4068, 0.3234, 0.2220, 0.0441, 0.0007, 0.0005, 0.0004, 0.0007, 0.0008,0.0005]
lambda = 5 - [0.3406, 0.2901, 0.2107, 0.1554, 0.0005, 0.0004, 0.0004, 0.0007, 0.0006,0.0006]
lambda = 10 - [4.1022e-01, 2.7880e-01, 1.6249e-01, 1.4537e-01, 4.9389e-04, 3.6017e-04, 3.4229e-04, 6.9533e-04, 4.8819e-04, 7.4599e-04]

In [175]:
#code that tests how well a discriminator operates

def assess_discriminator(discriminator, 
                         discriminator_optim,
                         dataset, 
                         groundtruth_weights, 
                         biased_weights,
                         Loss_function,
                         subset_size=2,
                         num_epochs = 1000,
                         device=torch.device('cuda:0'),
                         ):
    '''
    Given the input, it assesses how well a discriminator object can differentiate between two weightings of a singular data set. 

    Returns two loss np arrays in the form of a dictionary, results:
        results['truth'] = np array
        results['bias] = np array

    Input:
        discriminator - discriminator object
        discriminator_optim - discriminator optimizer
        dataset - np array size (n x d), where n = number of points, d = dimensionality
        groundtruth_weights - np array, sum(weights_one) = 1, groundtruth_weights.shape[0] = n 
        biased_weights - np array, sum(weights_two) = 1, biased_weights.shape[0] = n 
        subset_size - int, the size of the sample to take from the data set
        num_epochs - number of training iterations
    
    Output: results - dict
    '''
    results = {}
    results['truth'] = np.zeros(num_epochs)
    results['bias'] = np.zeros(num_epochs)
    for ne in range(num_epochs):
        #get data sample using groundtruth_weights
        groundtruth_indexes = np.random.choice(np.arange(dataset.shape[0]),size=subset_size,p=groundtruth_weights)
        groundtruth_onehot = np.zeros((subset_size, dataset.shape[0]))
        groundtruth_onehot[np.arange(subset_size),groundtruth_indexes] = 1
        groundtruth_sample = torch.tensor((groundtruth_onehot @ dataset),dtype=torch.float32).to(device).flatten()

        #get data sample using biased_weights
        biased_indexes = np.random.choice(np.arange(dataset.shape[0]),size=subset_size,p=groundtruth_weights)
        biased_onehot = np.zeros((subset_size, dataset.shape[0]))
        biased_onehot[np.arange(subset_size),biased_indexes] = 1
        biased_sample = torch.tensor((biased_onehot @ dataset),dtype=torch.float32).to(device).flatten()

        #pass both from discriminator and generate losses
        truth_prediction = discriminator(groundtruth_sample)
        biased_prediction = discriminator(biased_sample)
        loss_real = L(truth_prediction.to(device), torch.ones_like(truth_prediction).to(device))
        loss_fake = L(biased_prediction.to(device), torch.zeros_like(biased_prediction).to(device))
        discriminator_loss = loss_real+loss_fake
        #update discriminator
        discriminator_optim.zero_grad()
        discriminator_loss.backward(retain_graph=True)
        discriminator_optim.step()
        #store loss in arrays
        results['truth'][ne] = loss_real.detach().cpu().item()
        results['bias'][ne] = loss_real.detach().cpu().item()
    return results

SUBSET_SIZE = 10
DATA_DIM = 10
discriminator = DataDisctiminator(subset_size=SUBSET_SIZE,data_dimension=DATA_DIM)
discriminator_optimizer = torch.optim.Adam(discriminator.parameters(), lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
L = nn.BCEWithLogitsLoss()

loss_dict = assess_discriminator(discriminator, 
                     discriminator_optimizer,
                    dataset.groundtruth_data, 
                    groundtruth_weights=[0.1 for _ in range(dataset.n_truth)], 
                    biased_weights=[0.1 for _ in range(dataset.n_truth)],
                    Loss_function = L,
                    subset_size=SUBSET_SIZE,
                    num_epochs = 100,
                    device=device,
                    )